# Natural Language Inference

## Evaluating an ESIM-lite BiLSTM Attention Model

This notebook evaluates the trained Natural Language Inference (NLI) model on the development dataset. The same tokenizer and configuration used during training are loaded to ensure consistent preprocessing. The notebook reports evaluation metrics such as accuracy, precision, recall, and F1 score on the development set.

## Path

This section defines the file paths for the development dataset and the saved training outputs. These paths are used to load the trained model and evaluate it on the development set.

In [28]:
import os

# Path to the development dataset (used for evaluation)
BASE_DIR = ".."

DATA_DIR = os.path.join(BASE_DIR, "data")
DEV_CSV   = os.path.join(DATA_DIR, "dev.csv")

# Output directory containing the saved training artifacts
OUT_DIR  = os.path.join(BASE_DIR, "outputs/category_B")

print("DEV_CSV:", DEV_CSV)
print("OUT_DIR:", OUT_DIR)

DEV_CSV: ./data/dev.csv
OUT_DIR: ./outputs/category_B


## Imports

The required libraries are imported for data processing, loading the trained model, tokenising text, and computing evaluation metrics

In [29]:
# Standard libraries for data handling and model loading
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

# Evaluation metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# Used to pad token sequences to fixed length
from tensorflow.keras.preprocessing.sequence import pad_sequences

## Load Configuration

The configuration file saved during training is loaded to ensure that the same sequence length and preprocessing settings are used during evaluation.

In [30]:
# Load training configuration saved during training
cfg_path = os.path.join(OUT_DIR, "config.json")
with open(cfg_path, "r") as f:
    cfg = json.load(f)

# Use the same sequence length as used during training
MAX_LEN = int(cfg["MAX_LEN"])

print("Loaded config:", cfg_path)
print("MAX_LEN:", MAX_LEN)

Loaded config: ./outputs/category_B/config.json
MAX_LEN: 35


## Load Tokenizer

The tokenizer created during training is loaded so that the same vocabulary mapping is used when converting text into sequences.

In [31]:
# Load the tokenizer used during training
tok_path = os.path.join(OUT_DIR, "tokenizer.pkl")

with open(tok_path, "rb") as f:
    tokenizer = pickle.load(f)

print("Loaded tokenizer:", tok_path)

Loaded tokenizer: ./outputs/category_B/tokenizer.pkl


## Tokenisation Function

This function converts text into sequences of token IDs using the trained tokenizer and pads them to the fixed maximum sequence length.

In [32]:
# Convert text to token sequences and pad to MAX_LEN
def tok_pad(series):

    # Convert words to integer token IDs
    seqs = tokenizer.texts_to_sequences(series.astype(str).tolist())

    # Pad sequences so they all have the same length
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

## Load Model

The best model saved during training is loaded so it can be used for evaluation on the development set.

In [33]:
# Path to the best model checkpoint saved during training
model_path = os.path.join(OUT_DIR, "best_model.keras")

# Load the trained model
model = tf.keras.models.load_model(model_path, compile=False)

print("Loaded model:", model_path)

Loaded model: ./outputs/category_B/best_model.keras


## Evaluate on dev set

The model is evaluated on the development dataset to measure performance using accuracy, precision, recall, and F1 score.

In [34]:
# Load development dataset (contains labels)
dev_df = pd.read_csv(DEV_CSV)

# Convert text into padded sequences
X_dev_p = tok_pad(dev_df["premise"])
X_dev_h = tok_pad(dev_df["hypothesis"])

# True labels
y_dev = dev_df["label"].astype(int).values

# Predict probabilities
dev_probs = model.predict([X_dev_p, X_dev_h], batch_size=256, verbose=0).ravel()

# Convert probabilities to binary predictions using threshold 0.5
dev_pred = (dev_probs >= 0.5).astype(int)

# Compute evaluation metrics
acc = accuracy_score(y_dev, dev_pred)

# Macro metrics
macro_p  = precision_score(y_dev, dev_pred, average="macro")
macro_r  = recall_score(y_dev, dev_pred, average="macro")
macro_f1 = f1_score(y_dev, dev_pred, average="macro")

# Weighted metrics
weighted_p  = precision_score(y_dev, dev_pred, average="weighted")
weighted_r  = recall_score(y_dev, dev_pred, average="weighted")
weighted_f1 = f1_score(y_dev, dev_pred, average="weighted")

# Support
support = len(y_dev)

# Print metrics
print(f"DEV Accuracy            : {acc:.4f}")
print()
print(f"DEV Macro Precision     : {macro_p:.4f}")
print(f"DEV Macro Recall        : {macro_r:.4f}")
print(f"DEV Macro F1            : {macro_f1:.4f}")
print()
print(f"DEV Weighted Precision  : {weighted_p:.4f}")
print(f"DEV Weighted Recall     : {weighted_r:.4f}")
print(f"DEV Weighted F1         : {weighted_f1:.4f}")
print(f"DEV Support             : {support}\n")

print(classification_report(y_dev, dev_pred, digits=4))

DEV Accuracy            : 0.7213

DEV Macro Precision     : 0.7211
DEV Macro Recall        : 0.7206
DEV Macro F1            : 0.7208

DEV Weighted Precision  : 0.7213
DEV Weighted Recall     : 0.7213
DEV Weighted F1         : 0.7212
DEV Support             : 6736

              precision    recall  f1-score   support

           0     0.7178    0.6986    0.7080      3258
           1     0.7245    0.7427    0.7335      3478

    accuracy                         0.7213      6736
   macro avg     0.7211    0.7206    0.7208      6736
weighted avg     0.7213    0.7213    0.7212      6736

